# Notebook 1: Preprocesamiento de Datos e Ingeniería de Características (Fase II)

Este cuaderno automatiza la limpieza, imputación y escalamiento de los datos utilizando `Pipeline` y `ColumnTransformer` de Scikit-Learn.

### Justificaciones de Diseño:
1. **Manejo de Nulos:** Se utiliza `KNNImputer` para estimar los nulos experimentales basándose en el perfil de las proteínas más cercanas en el espacio euclidiano.
2. **Tratamiento de Outliers Macromoleculares:** Basado en la Fase I, `SEQUENCE_LENGTH` presenta asimetría positiva extrema por macrocomplejos legítimos. Se usa `RobustScaler` (basado en mediana e IQR) para evitar que estos extremos distorsionen la varianza del modelo.
3. **Codificación Categórica:** Se aplica `OneHotEncoder(handle_unknown='ignore')` para evitar fallos si aparecen nuevas cepas o taxonomías en producción.
4. **Prevención de Data Leakage:** El procesamiento se empaqueta de forma que el ajuste (`fit`) ocurra exclusivamente sobre los datos de entrenamiento.

* Carga de Dataset postEDA

In [ ]:
import pandas as pd

print("=== CARGANDO DATOS DESDE EL HITO POST-EDA ===")

# Ruta exacta donde se guardó el progreso del EDA
ruta_post_eda = "../selected_dataset/dataset4_enzimas_post_eda.csv"

try:
    df = pd.read_csv(ruta_post_eda)
    print(f"✔️ Conexión exitosa con el hito modular.")
    print(f"Dimensiones listas para procesar: {df.shape[0]} filas y {df.shape[1]} columnas.")
except FileNotFoundError:
    print(f"❌ No se encontró el archivo en la ruta '{ruta_post_eda}'.")
    print("👉 Verifica que la carpeta 'selected_dataset' exista y contenga el archivo generado.")

In [ ]:
# Celda de diagnóstico rápido
print("Las columnas reales en el DataFrame son:")
print(list(df.columns))

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

print("=== CONSTRUYENDO PIPELINE DE PREPROCESAMIENTO OPTIMIZADO (ANTI-MEMORY ERROR) ===")

# Separación de predictores y variable objetivo
y = df['LABEL'] 

columnas_remover = ['SEQUENCE_ID', 'SEQUENCE', 'AMBIGUOUS_COUNT', 'LABEL']
X = df.drop(columns=[col for col in columnas_remover if col in df.columns])

# Partición con Estratificación Estricta (20% para Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Identificación automática de tipos de columnas
columnas_numericas = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Seleccionamos explícitamente tanto 'object' como 'string' y categorías para blindar el código
columnas_categoricas = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

print(f"🔹 Columnas numéricas a procesar: {columnas_numericas}")
print(f"🔹 Columnas categóricas a procesar: {columnas_categoricas}")

# Configuración de sub-pipelines específicos

pipeline_numerico = Pipeline(steps=[
    ('imputador_num', SimpleImputer(strategy='median')), 
    ('escalador', RobustScaler())
])

# Permitimos 'sparse_output=True' para optimizar la memoria eficientemente
pipeline_categorico = Pipeline(steps=[
    ('imputador_cat', SimpleImputer(strategy='most_frequent')), 
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)) 
])

# Ensamble maestro (ColumnTransformer)
preprocesador = ColumnTransformer(transformers=[
    ('num', pipeline_numerico, columnas_numericas),
    ('cat', pipeline_categorico, columnas_categoricas)
])

# 6. Prueba de Control operativa en VS Code
print("\nAjustando y transformando los datos de entrenamiento...")
X_train_transformado = preprocesador.fit_transform(X_train)

print(f"\n✔️ ¡Éxito rotundo! El consumo de memoria bajó de 3.3 TB a unos pocos Megabytes.")
print(f"Dimensiones reales del set transformado: {X_train_transformado.shape}")

* Guardar proceso

In [ ]:
import joblib

print("=== GUARDANDO DATOS TRANSFORMAOS PARA EL NOTEBOOK 2 ===")

# Transformamos los conjuntos completos usando el pipeline que creamos
X_train_trans = preprocesador.fit_transform(X_train)
X_test_trans = preprocesador.transform(X_test)

# Guardamos las matrices y las etiquetas directamente en tu carpeta de datos
joblib.dump(X_train_trans, "../selected_dataset/X_train_trans.pkl")
joblib.dump(X_test_trans, "../selected_dataset/X_test_trans.pkl")
joblib.dump(y_train, "../selected_dataset/y_train.pkl")
joblib.dump(y_test, "../selected_dataset/y_test.pkl")

print("✔️ ¡Todo guardado con éxito! El Notebook 1 quedó 100% cerrado.")